In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [6]:
def generate_synthetic_spectra(df, target_class, num_synthetic_samples=10, alpha=0.5):
    """
    Generates synthetic spectra by creating linear combinations of pairs 
    from the same material class using the 'spectrums' DataFrame format:
    Columns = (Spectrum_ID, Labels, 4000.0, 3999.0, 3998.0, ..., 550.0).
    
    Parameters:
    - df: The 'spectrums' DataFrame
    - target_class: The string name of the material in the 'Labels' column to augment
    - num_synthetic_samples: How many new spectra to create
    - alpha: Controls the blend. If alpha=0.5, it's a 50/50 mix.
    """
    # get target class spectra
    class_df = df[df['Labels'] == target_class]
    
    if len(class_df) < 2: # input check
        raise ValueError("Need at least two spectra in the target class to generate synthetic samples.")
    
    wavenumber_cols = df.columns.drop(['Spectrum_ID', 'Labels'])
    synthetic_spectra = []
    next_id = df['Spectrum_ID'].max() + 1

    
    for i in range(num_synthetic_samples):
        # selects two (different) random spectra from the target class
        samples = class_df[wavenumber_cols].sample(2, replace=False)
        spectrum1 = samples.iloc[0].values.astype(float)
        spectrum2 = samples.iloc[1].values.astype(float)
        
        # creates a new spectrum by linearly combining the two spectra
        new_spectrum = alpha * spectrum1 + (1 - alpha) * spectrum2
        
        new_row = {'Spectrum_ID': next_id + i, 'Labels': target_class}

        for wave_col, intensity in zip(wavenumber_cols, new_spectrum):
            new_row[wave_col] = intensity
        
        # Append the new spectrum with its label to the list
        synthetic_spectra.append(new_row)
    
    # Convert the list of synthetic spectra to a DataFrame
    synthetic_df = pd.DataFrame(synthetic_spectra, columns=df.columns)
    
    return synthetic_df

In [11]:
# copy pasted from main jupyter notebook
df = pd.read_csv("02_feature_matrix_raw_T.csv", header = 0)

dataFrame = df.drop(columns=["Sample_ID", "Replica", "Origin", "Type"])

labels = (["Cotton"]*14 + ["Acrylic"]*18 + ["Nylon"]*18 + ["PP/PE"]*8 + ["Polyester"]*17 + ["Silk"]*11 + ["Wool"]*23)

ranges = [(24, 38), (50, 68), (70, 88), (88, 96), (96, 113), (125, 136), (137, 160)]
indices = np.concatenate([np.arange(s, e) for s, e in ranges])
spectrums_labeled = dataFrame.iloc[indices].reset_index(drop=True).copy()
spectrums_labeled.insert(loc = 2, column = "Labels", value = labels)
spectrums = spectrums_labeled.drop(columns=["Subtype"])
spectrums.iloc[:, :15].head(10)


,Spectrum_ID,Labels,4000.0,3999.0,3998.0,3997.0,3996.0,3995.0,3994.0,3993.0,3992.0,3991.0,3990.0,3989.0,3988.0
0,25,Cotton,99.10,99.11,99.12,99.11,99.10,99.09,99.06,99.04,99.03,99.03,99.04,99.05,99.05
1,26,Cotton,99.26,99.28,99.30,99.31,99.32,99.32,99.31,99.30,99.29,99.28,99.27,99.26,99.24
2,27,Cotton,99.89,99.89,99.89,99.89,99.89,99.89,99.90,99.91,99.92,99.94,99.95,99.95,99.95
3,28,Cotton,99.59,99.60,99.61,99.62,99.61,99.60,99.60,99.60,99.60,99.60,99.60,99.60,99.59
4,29,Cotton,100.71,100.72,100.74,100.75,100.77,100.77,100.76,100.75,100.73,100.72,100.71,100.70,100.69
5,30,Cotton,99.95,99.96,99.96,99.95,99.94,99.92,99.90,99.89,99.88,99.86,99.85,99.85,99.86
6,31,Cotton,101.33,101.33,101.34,101.35,101.36,101.36,101.37,101.37,101.37,101.37,101.35,101.34,101.32
7,32,Cotton,100.02,100.02,100.02,100.03,100.03,100.03,100.02,100.02,100.01,100.00,99.97,99.95,99.94
8,33,Cotton,100.45,100.47,100.48,100.49,100.48,100.47,100.46,100.45,100.45,100.46,100.46,100.46,100.45
9,34,Cotton,100.07,100.07,100.06,100.04,100.02,100.00,99.99,99.99,99.99,100.00,100.01,100.01,100.01


In [15]:
synthetic_silk = generate_synthetic_spectra(spectrums, target_class="Silk", num_synthetic_samples=20)
print(synthetic_silk.shape)
synthetic_silk.head()

(20, 3453)


,Spectrum_ID,Labels,4000.0,3999.0,3998.0,3997.0,3996.0,3995.0,3994.0,3993.0,...,559.0,558.0,557.0,556.0,555.0,554.0,553.0,552.0,551.0,550.0
0,161,Silk,99.450,99.440,99.425,99.410,99.405,99.410,99.420,99.420,...,82.825,83.060,82.895,82.235,81.375,80.650,80.125,79.740,79.610,79.660
1,162,Silk,98.360,98.365,98.370,98.380,98.380,98.385,98.385,98.370,...,72.800,72.480,71.800,70.920,70.115,69.565,69.260,69.165,69.240,69.350
2,163,Silk,98.465,98.475,98.490,98.505,98.515,98.530,98.530,98.525,...,66.180,65.670,64.950,64.255,63.910,63.920,63.920,63.625,63.190,62.840
3,164,Silk,101.360,101.360,101.355,101.345,101.345,101.345,101.355,101.350,...,82.120,82.270,82.315,82.195,82.105,82.135,82.115,81.890,81.615,81.460
4,165,Silk,100.300,100.305,100.320,100.330,100.330,100.335,100.330,100.325,...,77.310,77.045,76.840,76.665,76.520,76.400,76.245,76.050,75.880,75.735
